# Run TempME

Run this notebook for the selected `DATASET` / `MODEL` combo.

This notebook:
- Runs the official TempME submodule workflow for the selected `DATASET` / `MODEL` combo.
- Loads the matching checkpoint and explain index for the chosen combo.
- Saves official run artifacts under `resources/results/official_tempme/` and finishes with the shared one-spot metrics summary.

Usage:
- Change `DATASET` and `MODEL` in the first code cell when needed.
- Run the notebook top to bottom.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

_PROJECT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _PROJECT_CANDIDATES if (p / "I_explainer_benchmark").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root from the current working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from I_explainer_benchmark.src.notebooks.explainer_notebook_setup import (
    initialize_explainer_notebook,
    prepare_explainer_run,
)

# Select the dataset / model combo here.
DATASET = "simulate_v1"
MODEL = "tgn"

# Shared notebook setup. Leave the rest unchanged.
BOOT = initialize_explainer_notebook("07_tempme.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)
NB = BOOT.env
CFG = BOOT.config
SETTINGS = BOOT.settings
BENCHMARK_ROOT = BOOT.bench_root
REPO_ROOT = BOOT.repo_root
TEMP_ME_ROOT = BOOT.paths["TEMP_ME_ROOT"]


In [2]:
import sys
from pathlib import Path

import torch
from I_explainer_benchmark.src.notebooks.notebook_helpers import set_global_seed

BASE_TYPE = str(MODEL).strip().lower()
SUPPORTED_BASE_TYPES = set(SETTINGS["supported_base_types"])
if BASE_TYPE not in SUPPORTED_BASE_TYPES:
    raise ValueError(f"Unsupported MODEL={BASE_TYPE!r}. Choose one of {sorted(SUPPORTED_BASE_TYPES)}")

CTX = prepare_explainer_run("07_tempme.ipynb", dataset=DATASET, model=BASE_TYPE, start=PROJECT_ROOT)

DATASET_NAME = CTX.dataset
SEED = int(CFG["seed"])
set_global_seed(SEED)

QUICK_RUN = bool(SETTINGS["quick_run"])
FORCE_RERUN_BASE = bool(SETTINGS["force_rerun_base"])
FORCE_RERUN_EXPLAINER = bool(SETTINGS["force_rerun_explainer"])
TEMP_ME_TEST_THRESHOLD = SETTINGS.get("tempme_test_threshold")
TGAT_TEMPME_REQUIRED_LAYERS = int(SETTINGS.get("expected_tgat_layers", 2))

_graphs_python = Path("/Users/juliawenkmann/miniconda3/envs/graphs/bin/python")
PYTHON_BIN = str(_graphs_python if _graphs_python.exists() else Path(sys.executable))
CUDA_AVAILABLE = bool(torch.cuda.is_available())
DEFAULT_GPU = 0 if CUDA_AVAILABLE else -1

BASE_OVERRIDES_QUICK = dict(SETTINGS["base_overrides_quick"])
EXPLAINER_OVERRIDES_QUICK = dict(SETTINGS["explainer_overrides_quick"])
BASE_OVERRIDES_FULL = dict(SETTINGS["base_overrides_full"])
EXPLAINER_OVERRIDES_FULL = dict(SETTINGS["explainer_overrides_full"])
TEMPME_DEGREE_BY_DATASET = dict(SETTINGS["tempme_degree_by_dataset"])
BASE_OVERRIDES = dict(BASE_OVERRIDES_QUICK if QUICK_RUN else BASE_OVERRIDES_FULL)
EXPLAINER_OVERRIDES = dict(EXPLAINER_OVERRIDES_QUICK if QUICK_RUN else EXPLAINER_OVERRIDES_FULL)
FORCE_RERUN_BASE_EFFECTIVE = bool(FORCE_RERUN_BASE)
FORCE_RERUN_EXPLAINER_EFFECTIVE = bool(FORCE_RERUN_EXPLAINER)

N_EVAL_EVENTS = int(SETTINGS["n_eval_events"])
CANDIDATES_SIZE = int(SETTINGS["candidates_size"])
USE_CACHED_TEMPME_RESULTS = bool(SETTINGS["use_cached_tempme_results"])

EXPLAIN_INDEX_PATH = CTX.explain_index_path
CKPT_PATH = CTX.checkpoint_path
DEVICE = CTX.device
TEMP_ME_MODEL_ROOT = BENCHMARK_ROOT / "resources" / "model" / "tempme" / DATASET_NAME / BASE_TYPE
BASE_CKPT = TEMP_ME_MODEL_ROOT / "tgnn" / f"{BASE_TYPE}_{DATASET_NAME}.pt"
EXPL_CKPT = TEMP_ME_MODEL_ROOT / "explainer" / BASE_TYPE / f"{DATASET_NAME}.pt"

print("DATASET:", DATASET_NAME)
print("MODEL  :", BASE_TYPE)
print("CKPT   :", CKPT_PATH)
print("DEVICE :", DEVICE)


DATASET: simulate_v1
MODEL  : tgn
CKPT   : /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/models/simulate_v1/tgn/tgn_simulate_v1_best.pth
DEVICE : mps


In [3]:
from IPython.display import display

from I_explainer_benchmark.src.notebooks.explainer_notebook_runtime import (
    TempMENotebookRuntimeConfig,
    run_tempme_training_pipeline,
)

_tempme_runtime = run_tempme_training_pipeline(
    TempMENotebookRuntimeConfig(
        project_root=PROJECT_ROOT,
        benchmark_root=BENCHMARK_ROOT,
        temp_me_root=TEMP_ME_ROOT,
        dataset_name=DATASET_NAME,
        base_type=BASE_TYPE,
        python_bin=PYTHON_BIN,
        quick_run=QUICK_RUN,
        base_overrides=dict(BASE_OVERRIDES),
        explainer_overrides=dict(EXPLAINER_OVERRIDES),
        force_rerun_base_effective=FORCE_RERUN_BASE_EFFECTIVE,
        force_rerun_explainer_effective=FORCE_RERUN_EXPLAINER_EFFECTIVE,
        base_ckpt=BASE_CKPT,
        expl_ckpt=EXPL_CKPT,
    )
)

base_logs = _tempme_runtime.base_logs
base_metrics = _tempme_runtime.base_metrics
expl_logs = _tempme_runtime.expl_logs
expl_metrics = _tempme_runtime.expl_metrics
metrics_df = _tempme_runtime.metrics_df
metrics_path = _tempme_runtime.metrics_path
logs_path = _tempme_runtime.logs_path

display(metrics_df)
print('Saved metrics to:', metrics_path)
print('Saved logs to:', logs_path)


Skipping learn_base.py (checkpoint exists): /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/model/tempme/simulate_v1/tgn/tgnn/tgn_simulate_v1.pt
Skipping temp_exp_main.py (checkpoint exists): /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/model/tempme/simulate_v1/tgn/explainer/tgn/simulate_v1.pt
Loaded cached temp_exp_main metrics from: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/explainer/tempme/simulate_v1/tgn/runs/latest_temp_exp_main.json


,stage,testing_epoch,testing_loss,testing_aps,testing_auc,testing_acc,testing_fidelity_prob,testing_fidelity_logit,ratio_aps,ratio_auc,ratio_acc,ratio_prob,ratio_logit,acc_auc
0,temp_exp_main,0.0,0.712732,0.988306,0.969952,0.648193,0.000248,0.001005,0.997277,0.993278,0.976135,-0.000095,-0.00038,0.976135


Saved metrics to: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/explainer/tempme/simulate_v1/tgn/runs/metrics_20260315_200648.csv
Saved logs to: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/explainer/tempme/simulate_v1/tgn/runs/logs_20260315_200648.json


In [4]:
# Run/reuse TempME explanations and persist results.jsonl for downstream metrics
from pathlib import Path

import torch

from I_explainer_benchmark.src.explainers.builder import make_explainer_builder
from I_explainer_benchmark.src.notebooks.explainer_notebook_runtime import (
    prepare_standard_edge_explainer_run,
    run_or_reuse_explanations,
)
from I_explainer_benchmark.src.notebooks.notebook_helpers import forced_target_event_ids_for_combo as _forced_target_event_ids_for_combo

SEED = 42
N_EVAL_EVENTS = 30
CANDIDATES_SIZE = 70
USE_CACHED_TEMPME_RESULTS = True

EXPLAIN_INDEX_PATH = BENCHMARK_ROOT / 'resources' / 'explainer' / 'explain_index' / f'{DATASET_NAME}.csv'
CHECKPOINT_CANDIDATES = [
    BENCHMARK_ROOT / 'resources' / 'models' / DATASET_NAME / BASE_TYPE / f'{BASE_TYPE}_{DATASET_NAME}_best.pth',
    BENCHMARK_ROOT / 'resources' / 'models' / DATASET_NAME / 'checkpoints' / f'{BASE_TYPE}_{DATASET_NAME}_best.pth',
    BENCHMARK_ROOT / 'resources' / 'models' / 'checkpoints' / f'{BASE_TYPE}_{DATASET_NAME}_best.pth',
]
CKPT_PATH = next((p for p in CHECKPOINT_CANDIDATES if p.exists()), None)
if CKPT_PATH is None:
    raise FileNotFoundError(
        'Could not find checkpoint. Checked\n' + '\n'.join(str(p) for p in CHECKPOINT_CANDIDATES)
    )

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FORCED_TARGET_EVENT_IDS = list(
    globals().get('FORCED_TARGET_EVENT_IDS', _forced_target_event_ids_for_combo(DATASET_NAME, BASE_TYPE))
)

build_explainer = make_explainer_builder(
    dataset_name=DATASET_NAME,
    model_type=BASE_TYPE,
    device=DEVICE,
    seed=SEED,
    callable_scope=globals(),
)

tempme_explainer = build_explainer('tempme', allow_missing=False)
print('Explainer alias:', tempme_explainer.alias)

official_tempme_root = BENCHMARK_ROOT / 'resources' / 'results' / 'official_tempme'
official_tempme_dir = official_tempme_root / str(BASE_TYPE)
official_tempme_dir.mkdir(parents=True, exist_ok=True)

ctx = prepare_standard_edge_explainer_run(
    dataset_name=DATASET_NAME,
    model_name=BASE_TYPE,
    ckpt_path=CKPT_PATH,
    device=DEVICE,
    explain_index_path=EXPLAIN_INDEX_PATH,
    n_eval_events=N_EVAL_EVENTS,
    out_dir=official_tempme_dir,
    explainer=tempme_explainer,
    candidates_size=CANDIDATES_SIZE,
    seed=SEED,
    include_event_ids=FORCED_TARGET_EVENT_IDS,
)
model = ctx.model
events = ctx.events
all_event_idxs = ctx.all_event_idxs
target_event_idxs = ctx.target_event_idxs
anchors = ctx.anchors

print('CKPT_PATH:', CKPT_PATH)
print('DEVICE:', DEVICE)
print('Anchors selected:', len(anchors), '/', len(all_event_idxs))
if anchors:
    print('First target event_idx:', anchors[0]['event_idx'])

out, run_dir, out_jsonl, base_name = run_or_reuse_explanations(
    runner=ctx.runner,
    anchors=anchors,
    dataset_name=DATASET_NAME,
    model_name=BASE_TYPE,
    explainer_name='tempme',
    target_event_idxs=target_event_idxs,
    use_cached=USE_CACHED_TEMPME_RESULTS,
    cache_roots=[official_tempme_dir, official_tempme_root],
)

print('run_dir:', run_dir)
print('out_jsonl:', out_jsonl)


Built explainer 'tempme' from /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/configs/explainer/tempme.json
Explainer alias: tempme
102 events to explain
CKPT_PATH: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/models/simulate_v1/tgn/tgn_simulate_v1_best.pth
DEVICE: cpu
Anchors selected: 30 / 102
First target event_idx: 3
Using cached tempme explanations from: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_tempme/tgn/simulate_v1_tgn_official_tempme_20260313_161842
run_dir: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_tempme/tgn/simulate_v1_tgn_official_tempme_20260313_161842
out_jsonl: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benc

In [5]:
# Shared metrics + export pipeline (clean notebook wrapper)
from I_explainer_benchmark.src.notebooks.notebook_metrics_pipeline import run_notebook_metrics_from_namespace

_pipeline_out = run_notebook_metrics_from_namespace(globals(), CFG)
globals().update(_pipeline_out)
run_dir_export = _pipeline_out['run_dir_export']
export_root = _pipeline_out['export_root']
out_jsonl = _pipeline_out['out_jsonl']
out_dir = _pipeline_out['out_dir']
base_name = _pipeline_out['base_name']
print('CURRENT_RUN_ID:', _pipeline_out['CURRENT_RUN_ID'])
print('Saved run export directory:', run_dir_export)
print('Updated global tables under:', export_root)


/Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/src/notebooks/notebook_runtime_common.py:39: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  prev = pd.read_csv(path)


CURRENT_RUN_ID: simulate_v1_tgn_official_tempme_20260313_161842
Saved run export directory: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready/tgn/simulate_v1_tgn_official_tempme_20260313_161842
Updated global tables under: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready


In [6]:
# One-spot metrics summary (official)
from I_explainer_benchmark.src.notebooks.notebook_display import show_one_spot_metrics_summary

_one_spot = globals().get("one_spot", globals().get("_pipeline_out", {}).get("one_spot", {}))
show_one_spot_metrics_summary(_one_spot)


One-spot metrics summary (official):


,dataset,model,explainer,variant,n_events,best_fid,best_fid_sparsity,aufsc,best_minus_aufsc,best_fid_raw,...,tempme_acc_auc.ratio_acc,tempme_acc_auc.ratio_prob,tempme_acc_auc.ratio_logit,tempme_acc_auc.ratio_aps,tempme_acc_auc.ratio_auc,temgx_aufsc,cody_AUFSC_plus,cody_AUFSC_minus,cody_CHARR,elapsed_sec
0,simulate_v1,tgn,tempme,official,30,1.1441190203,1.0,-0.307790435,1.4519094553,1.1441190203,...,0.7197916667,-0.0217003194,-0.2037920761,0.8322916667,0.9375,0.1091120743,0.6748701771,0.4259799343,0.5222893666,0.0070840737
